# Results from CSV (match `check.ipynb` layout)

Reads `base.csv` / `ensemble.csv` / `tabm.csv` per dataset under `RESULTS_ROOT/{dataset_dir}/`.

For each **dataset**, **method** (BASE / ENS / TABM), **architecture** (`model`), and **split**:
- Among all rows (all `num_layers` × `hidden_dim` × `lr`), pick the row with **highest `val_metric`** (same objective as training / early stopping).
- Report **test** score: **`test_roc_auc`** for `questions` / `minesweeper` / `tolokers`, else **`test_acc`** (same rule as `check.ipynb`).

Then **mean ± std** over splits (0–9) and pivot like `check.ipynb`.

In [2]:
import pandas as pd
from pathlib import Path
from typing import Optional

RESULTS_ROOT = Path('results_merged')

# models = ['GT-sep', 'GAT-sep', 'GAT', 'TAG', 'GCN', 'SAGE', 'GT', 'ResNet']
models = ['GT-sep', 'GAT-sep', 'GAT', 'GCN', 'SAGE', 'GT', 'ResNet']

ROC_PRIMARY_DATASETS = frozenset({'questions', 'minesweeper'})

# Keys = dataset id (as in check.ipynb); values = subdir name under RESULTS_ROOT
DATASET_DIRS = {
    'roman_empire': 'roman_empire',
    'amazon_ratings': 'amazon_ratings',
    'minesweeper': 'minesweeper',
    'questions': 'questions',
    'tolokers': 'tolokers',
}

METHOD_FILES = {
    'BASE': 'base.csv',
    'ENS': 'ensemble.csv',
    'TABM': 'tabm.csv',
}

NUM_SPLITS = 10


def primary_test_value(row: pd.Series, primary_is_roc: bool) -> Optional[float]:
    """Test metric to aggregate (aligned with check.ipynb)."""
    if primary_is_roc:
        v = row.get('test_roc_auc')
        if pd.notna(v):
            return float(v)
        return None
    v = row.get('test_acc')
    if pd.notna(v):
        return float(v)
    return None


def best_row_per_split(df: pd.DataFrame, model: str, split: int) -> Optional[pd.Series]:
    """Single best config for (model, split) by validation metric."""
    sub = df[(df['model'] == model) & (df['split'] == split)].copy()
    if sub.empty:
        return None
    sub = sub.dropna(subset=['val_metric'])
    if sub.empty:
        return None
    i = sub['val_metric'].idxmax()
    return sub.loc[i]


rows = []
for dataset_name, ds_dir in DATASET_DIRS.items():
    primary_is_roc = dataset_name in ROC_PRIMARY_DATASETS
    for method_name, csv_name in METHOD_FILES.items():
        csv_path = RESULTS_ROOT / ds_dir / csv_name
        if not csv_path.is_file():
            continue
        df = pd.read_csv(csv_path)
        if 'val_metric' not in df.columns:
            raise ValueError(f"Missing val_metric column in {csv_path}")

        for model in models:
            split_scores = []
            for split in range(NUM_SPLITS):
                row = best_row_per_split(df, model, split)
                if row is None:
                    continue
                pv = primary_test_value(row, primary_is_roc)
                if pv is not None:
                    split_scores.append(pv)

            if split_scores:
                rows.append({
                    'model_variant': f'{model}_{method_name}',
                    'dataset': dataset_name,
                    'mean_acc': float(pd.Series(split_scores).mean()),
                    'std_acc': float(pd.Series(split_scores).std(ddof=0)),
                })

if not rows:
    raise RuntimeError(
        f"No data found under {RESULTS_ROOT}. "
        "Set RESULTS_ROOT and ensure CSVs exist (base.csv / ensemble.csv / tabm.csv per dataset)."
    )

df_rows = pd.DataFrame(rows)

_col_order = list(DATASET_DIRS.keys())
mx_mean = df_rows.pivot_table(
    index='model_variant',
    columns='dataset',
    values='mean_acc',
    aggfunc='mean',
).sort_index().mul(100)
mx_mean = mx_mean.reindex(columns=[c for c in _col_order if c in mx_mean.columns])

mx_std = df_rows.pivot_table(
    index='model_variant',
    columns='dataset',
    values='std_acc',
    aggfunc='mean',
).reindex(mx_mean.index).mul(100)
mx_std = mx_std.reindex(columns=mx_mean.columns)

mx_text = mx_mean.round(3).astype(str) + ' ± ' + mx_std.round(3).astype(str)


def highlight_top2(col):
    styles = ['' for _ in col]
    non_na = col.dropna()
    if non_na.empty:
        return styles
    unique_vals = sorted(non_na.unique(), reverse=True)
    best = unique_vals[0]
    second = unique_vals[1] if len(unique_vals) > 1 else None
    for i, v in enumerate(col):
        if pd.isna(v):
            continue
        if v == best:
            styles[i] = 'background-color: #ffd700; font-weight: 700'
        elif second is not None and v == second:
            styles[i] = 'background-color: #c0c0c0; font-weight: 700'
    return styles


mx_text.style.apply(lambda col: highlight_top2(mx_mean[col.name]), axis=0)

dataset,roman_empire,amazon_ratings,minesweeper,questions,tolokers
model_variant,,,,,
GAT-sep_BASE,89.229 ± 0.578,54.431 ± 0.468,93.391 ± 0.479,77.573 ± 0.632,80.129 ± 0.887
GAT-sep_ENS,90.141 ± 0.42,55.584 ± 0.506,93.71 ± 0.414,78.045 ± 0.845,80.272 ± 0.878
GAT-sep_TABM,91.19 ± 0.327,55.765 ± 0.436,93.701 ± 0.459,79.099 ± 0.923,81.299 ± 0.726
GAT_BASE,82.783 ± 0.874,51.71 ± 0.512,92.355 ± 0.494,77.169 ± 0.673,81.19 ± 0.881
GAT_ENS,83.893 ± 0.399,53.021 ± 0.618,92.768 ± 0.583,77.995 ± 0.905,81.878 ± 0.444
GAT_TABM,82.711 ± 0.48,52.892 ± 0.442,91.106 ± 0.74,79.126 ± 0.885,81.884 ± 0.659
GCN_BASE,73.359 ± 0.732,50.304 ± 0.392,89.643 ± 0.534,76.329 ± 0.925,81.534 ± 0.795
GCN_ENS,75.245 ± 0.729,50.676 ± 0.474,89.903 ± 0.545,76.731 ± 1.08,81.524 ± 0.864
GCN_TABM,77.739 ± 0.542,49.977 ± 0.574,88.771 ± 0.62,77.502 ± 1.023,81.33 ± 0.661


In [ ]:
from collections import defaultdict

# Генерирует TeX-таблицу по mx_text (и выделяет лучшие/вторые результаты для каждого столбца)
def to_tex_table(mx_mean, mx_std, mx_text, col_names=None, caption=None, label=None):
    # col_names: необязательно, для переименования столбцов (словари)

    # Получаем названия моделей и датасетов
    model_variants = mx_text.index.tolist()
    datasets = mx_text.columns.tolist()

    # Тексты в таблице
    cell_strs = mx_text.values

    # Выделение лучших/вторых по каждому столбцу (датасету)
    best_indices_by_col = []
    second_indices_by_col = []
    for col_idx, d in enumerate(datasets):
        col_values = mx_mean.iloc[:, col_idx]
        na_mask = ~col_values.isna()
        values = col_values[na_mask]
        if len(values) == 0:
            best_indices_by_col.append(set())
            second_indices_by_col.append(set())
            continue
        sorted_unique = sorted(values.unique(), reverse=True)
        best_val = sorted_unique[0]
        sec_val = sorted_unique[1] if len(sorted_unique) > 1 else None
        best_indices = set(col_values.index[col_values == best_val])
        best_indices_by_col.append(best_indices)
        if sec_val is not None:
            second_indices = set(col_values.index[col_values == sec_val])
        else:
            second_indices = set()
        second_indices_by_col.append(second_indices)

    # Люди ждут подчеркивания для second best, жирного для best
    def highlight_cell(row_idx, col_idx, cell_text):
        model_key = mx_text.index[row_idx]
        # в best/second списках - по индексам
        if model_key in best_indices_by_col[col_idx]:
            return r"\textbf{" + cell_text + "}"
        elif model_key in second_indices_by_col[col_idx]:
            return r"\underline{" + cell_text + "}"
        else:
            return cell_text

    # Названия столбцов для TeX
    if col_names is not None:
        col_headers = [col_names.get(d, d) for d in datasets]
    else:
        col_headers = datasets

    # Эскапируем спецсимволы для таблички (названия моделей)
    def tex_esc(s):
        return str(s).replace('_', r'\_')

    lines = []
    lines.append(r"\begin{table*}[t]")
    lines.append(r"\centering")
    lines.append(r"\renewcommand{\arraystretch}{1.15}")

    if caption is not None:
        lines.append(rf"\caption{{{caption}}}")
    if label is not None:
        lines.append(rf"\label{{{label}}}")

    lines.append(r"\resizebox{\textwidth}{!}{")
    lines.append(r"\begin{tabular}{" + "l" + "c" * len(datasets) + "}")
    lines.append(r"\toprule")
    header = " & ".join(["\\textbf{Model variant}"] + [f"\\textbf{{{tex_esc(h)}}}" for h in col_headers]) + r" \\"
    lines.append(header)
    lines.append(r"\midrule")

    # Строки с данными
    for row_idx, mname in enumerate(model_variants):
        row_list = [tex_esc(mname)]
        for col_idx in range(len(datasets)):
            cell = cell_strs[row_idx][col_idx]
            # пустые значения - пусть будет "--"
            if pd.isna(cell) or cell == "nan ± nan":
                cell_tex = "--"
            else:
                cell_tex = highlight_cell(row_idx, col_idx, cell.replace('±', r'$\pm$'))
            row_list.append(cell_tex)
        lines.append(" & ".join(row_list) + r" \\")

    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")
    lines.append(r"}")  # end resizebox
    lines.append(r"\end{table*}")

    return "\n".join(lines)

# Пример: можно использовать примерно так
tex_caption = (
    "Results on additional benchmark datasets. "
    "Best result for each dataset is highlighted in bold, and the second-best result is underlined."
)
tex_label = "tab:additional_benchmarks"

# Если нужны переименования датасетов (чтобы было красиво)
pretty_dataset_names = {
    "roman_empire": "Roman Empire",
    "amazon_ratings": "Amazon Ratings",
    "minesweeper": "Minesweeper",
    "questions": "Questions",
    "tolokers": "Tolokers",
}

tex = to_tex_table(mx_mean, mx_std, mx_text, col_names=pretty_dataset_names, caption=tex_caption, label=tex_label)
print(tex)

In [7]:
data = pd.read_csv('results/minesweeper/base.csv')

In [8]:
data

,dataset,model,num_layers,hidden_dim,lr,split,best_step,train_metric_name,train_metric,train_acc,...,val_rec_macro,test_metric_name,test_metric,test_acc,test_loss,test_roc_auc,test_f1_macro,test_f1_weighted,test_prec_macro,test_rec_macro
0,minesweeper,ResNet,1,512,0.00003,0,30,roc_auc,0.517150,0.8000,...,NaN,roc_auc,0.520379,0.8000,0.503808,0.520379,NaN,NaN,NaN,NaN
1,minesweeper,ResNet,1,512,0.00003,1,40,roc_auc,0.507178,0.8000,...,NaN,roc_auc,0.525914,0.8000,0.512923,0.525914,NaN,NaN,NaN,NaN
2,minesweeper,ResNet,1,512,0.00003,3,10,roc_auc,0.496989,0.8000,...,NaN,roc_auc,0.519573,0.8000,0.553233,0.519573,NaN,NaN,NaN,NaN
3,minesweeper,ResNet,1,512,0.00003,2,30,roc_auc,0.513693,0.8000,...,NaN,roc_auc,0.514335,0.8000,0.508045,0.514335,NaN,NaN,NaN,NaN
4,minesweeper,ResNet,1,512,0.00003,4,10,roc_auc,0.521819,0.8000,...,NaN,roc_auc,0.523025,0.8000,0.528870,0.523025,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
395,minesweeper,TAG,5,512,0.00003,5,640,roc_auc,0.969878,0.9212,...,NaN,roc_auc,0.940287,0.8888,0.272366,0.940287,NaN,NaN,NaN,NaN
396,minesweeper,TAG,5,512,0.00003,6,510,roc_auc,0.957316,0.9028,...,NaN,roc_auc,0.926081,0.8684,0.292967,0.926081,NaN,NaN,NaN,NaN
397,minesweeper,TAG,5,512,0.00003,7,690,roc_auc,0.965584,0.9174,...,NaN,roc_auc,0.946203,0.8840,0.243702,0.946203,NaN,NaN,NaN,NaN
398,minesweeper,TAG,5,512,0.00003,8,600,roc_auc,0.962699,0.9090,...,NaN,roc_auc,0.941893,0.8924,0.261345,0.941893,NaN,NaN,NaN,NaN
